# CS781 Phase 2 — Efficient exact-character NER

This preserves the improved Phase 1 efficient weights, switches their local attention to bidirectional for NER, and predicts exact character boundaries. Attach the private unpacked improved-efficient dataset containing `efficient.pt` and `phase1_bpe.json`, enable GPU and Internet, and enable the Kaggle secret `HF`.


In [ ]:
!nvidia-smi
!pip install -q tokenizers huggingface_hub
import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF')
print('HF token loaded')


In [ ]:
from pathlib import Path
for name in ('efficient.pt', 'phase1_bpe.json'):
    matches = list(Path('/kaggle/input').rglob(name))
    print(name, matches)
    assert len(matches) == 1, f'Attach exactly one {name}'


In [ ]:
%%writefile /kaggle/working/phase1_pipeline.py
"""Phase 1 training, generation, validation, and submission pipeline."""

from __future__ import annotations

import argparse
import json
import math
import os
import random
import time
import zipfile
from dataclasses import asdict, dataclass
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F


SEED = 230338
SPECIAL = ["<pad>", "<bos>", "<eos>", "<unk>"]
MODEL_NAMES = ("decoder", "encoder_decoder", "hybrid", "efficient")
TRAIN_FILES = (
    "train_acts.json",
    "train_high_courts.json",
    "train_supreme_court.json",
    "train_tribunals.json",
)


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


@dataclass
class RunConfig:
    vocab_size: int = 4096
    d_model: int = 512
    n_heads: int = 8
    seq_len: int = 256
    batch_size: int = 2
    grad_accum: int = 4
    steps: int = 400
    learning_rate: float = 3e-4
    weight_decay: float = 0.1
    train_chars: int = 40_000_000
    train_tokens: int = 12_000_000
    tokenizer_chars: int = 10_000_000
    generation_batch_size: int = 8
    max_generation_tokens: int = 320
    target_words: int = 128


class TokenBlocks(torch.utils.data.Dataset):
    def __init__(self, ids: list[int], block: int, samples: int, seed: int):
        if len(ids) <= block:
            raise ValueError(f"Need more than {block} tokens; received {len(ids)}")
        self.ids = torch.tensor(ids, dtype=torch.long)
        generator = torch.Generator().manual_seed(seed)
        self.starts = torch.randint(0, len(ids) - block, (samples,), generator=generator)
        self.block = block

    def __len__(self) -> int:
        return len(self.starts)

    def __getitem__(self, index: int) -> torch.Tensor:
        start = int(self.starts[index])
        return self.ids[start : start + self.block]


def make_generation_validation(texts: list[str], examples: int = 8):
    """Hold out complete documents for the official 512-word to 128-word task."""
    training_texts, prompts, truth = [], {}, {}
    for text in texts:
        words = text.split()
        if len(truth) < examples and len(words) >= 640:
            example_id = f"heldout_{len(truth) + 1}"
            prompts[example_id] = {"text": " ".join(words[:512])}
            truth[example_id] = {"text": " ".join(words[512:640])}
        else:
            training_texts.append(text)
    if len(truth) < examples:
        raise RuntimeError(f"Could construct only {len(truth)} held-out continuation examples")
    return training_texts, prompts, truth


def exclude_generation_validation(texts: list[str], prompts: dict, truth: dict) -> list[str]:
    signatures = {
        example_id: f"{prompts[example_id]['text']} {truth[example_id]['text']}"
        for example_id in truth
    }
    filtered = [
        text for text in texts
        if " ".join(text.split()[:640]) not in signatures.values()
    ]
    if len(filtered) != len(texts) - len(truth):
        raise RuntimeError("Could not exclude every reused held-out document from training")
    return filtered


def train_tokenizer(texts: list[str], path: Path, vocab_size: int):
    """Train a new byte-level BPE on only the selected training corpus."""
    from tokenizers import Tokenizer, decoders, models, normalizers, pre_tokenizers, trainers

    tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
    tokenizer.normalizer = normalizers.NFC()
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=SPECIAL, min_frequency=2)
    tokenizer.train_from_iterator(texts, trainer=trainer)
    tokenizer.save(str(path))
    return tokenizer


def load_tokenizer(path: Path):
    from tokenizers import Tokenizer

    return Tokenizer.from_file(str(path))


def collect_training_text(config: RunConfig) -> list[str]:
    """Stream a balanced slice from the four official Phase 1 sources."""
    import ijson
    import requests

    texts, chars = [], 0
    quota = config.train_chars // len(TRAIN_FILES)
    headers = {}
    if os.environ.get("HF_TOKEN"):
        headers["Authorization"] = f"Bearer {os.environ['HF_TOKEN']}"
    for filename in TRAIN_FILES:
        url = (
            "https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone/resolve/main/"
            f"Phase_1_Pretraining/Train/{filename}"
        )
        source_chars = 0
        with requests.get(url, headers=headers, stream=True, timeout=(30, 900)) as response:
            response.raise_for_status()
            response.raw.decode_content = True
            for record in ijson.items(response.raw, "item"):
                text = record.get("text", "")
                if not isinstance(text, str) or not text.strip():
                    continue
                remaining = quota - source_chars
                texts.append(text[:remaining])
                taken = min(len(text), remaining)
                chars += taken
                source_chars += taken
                if source_chars >= quota:
                    break
        print(f"{filename}: {source_chars:,} characters")
    if chars < 100_000:
        raise RuntimeError("Official training stream returned too little text")
    print(f"Collected {chars:,} training characters in {len(texts):,} records")
    return texts


def tokenize_training_text(texts: list[str], tokenizer, config: RunConfig) -> list[int]:
    eos = tokenizer.token_to_id("<eos>")
    ids: list[int] = []
    for text in texts:
        ids.extend(tokenizer.encode(text).ids)
        ids.append(eos)
        if len(ids) >= config.train_tokens:
            break
    print(f"Prepared {len(ids):,} training tokens")
    return ids[: config.train_tokens]


def load_test_data() -> dict[str, dict[str, str]]:
    from huggingface_hub import hf_hub_download

    path = hf_hub_download(
        repo_id="Exploration-Lab/CS-781-Capstone",
        filename="Phase_1_Pretraining/Test/test.json",
        repo_type="dataset",
        token=os.environ.get("HF_TOKEN"),
    )
    with open(path, encoding="utf-8") as stream:
        data = json.load(stream)
    if not isinstance(data, dict) or not data:
        raise ValueError("Official test input is not a non-empty ID mapping")
    return data


def causal_mask(length: int, device: torch.device) -> torch.Tensor:
    return torch.triu(torch.ones(length, length, dtype=torch.bool, device=device), diagonal=1)


def sinusoidal_positions(length: int, dim: int, device: torch.device) -> torch.Tensor:
    position = torch.arange(length, device=device).float().unsqueeze(1)
    scale = torch.exp(torch.arange(0, dim, 2, device=device).float() * (-math.log(10000.0) / dim))
    output = torch.zeros(length, dim, device=device)
    output[:, 0::2] = torch.sin(position * scale)
    output[:, 1::2] = torch.cos(position * scale)
    return output


class GatedContextFusion(nn.Module):
    """Inject a global or causal summary through a learned content-dependent gate."""

    def __init__(self, dim: int):
        super().__init__()
        self.context = nn.Linear(dim, dim, bias=False)
        self.gate = nn.Linear(2 * dim, dim)

    def forward(self, hidden: torch.Tensor, summary: torch.Tensor) -> torch.Tensor:
        gate = torch.sigmoid(self.gate(torch.cat((hidden, summary), dim=-1)))
        return hidden + gate * self.context(summary)


class DecoderOnlyLM(nn.Module):
    def __init__(self, cfg: RunConfig, layers: int = 16):
        super().__init__()
        self.cfg = cfg
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
        layer = nn.TransformerEncoderLayer(
            cfg.d_model, cfg.n_heads, 4 * cfg.d_model, batch_first=True,
            norm_first=True, activation="gelu", dropout=0.0,
        )
        self.blocks = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.fusion = GatedContextFusion(cfg.d_model)
        self.norm = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.embedding.weight

    def logits(self, ids: torch.Tensor) -> torch.Tensor:
        x = self.embedding(ids) + sinusoidal_positions(ids.size(1), self.cfg.d_model, ids.device)
        x = self.blocks(x, mask=causal_mask(ids.size(1), ids.device))
        denominator = torch.arange(1, ids.size(1) + 1, device=ids.device).view(1, -1, 1)
        x = self.fusion(x, x.cumsum(dim=1) / denominator)
        return self.head(self.norm(x))

    def loss(self, batch: torch.Tensor, bos: int) -> torch.Tensor:
        del bos
        return F.cross_entropy(self.logits(batch[:, :-1]).flatten(0, 1), batch[:, 1:].flatten())

    def next_logits(self, source: torch.Tensor, generated: torch.Tensor, bos: int) -> torch.Tensor:
        del bos
        context = torch.cat((source, generated), dim=1)[:, -self.cfg.seq_len :]
        return self.logits(context)[:, -1]


class EncoderDecoderLM(nn.Module):
    def __init__(self, cfg: RunConfig, encoder_layers: int = 7, decoder_layers: int = 7):
        super().__init__()
        self.cfg = cfg
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
        encoder = nn.TransformerEncoderLayer(
            cfg.d_model, cfg.n_heads, 4 * cfg.d_model, batch_first=True,
            norm_first=True, activation="gelu", dropout=0.0,
        )
        decoder = nn.TransformerDecoderLayer(
            cfg.d_model, cfg.n_heads, 4 * cfg.d_model, batch_first=True,
            norm_first=True, activation="gelu", dropout=0.0,
        )
        self.encoder = nn.TransformerEncoder(encoder, encoder_layers, enable_nested_tensor=False)
        self.decoder = nn.TransformerDecoder(decoder, decoder_layers)
        self.fusion = GatedContextFusion(cfg.d_model)
        self.norm = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.embedding.weight

    def encode(self, source: torch.Tensor) -> torch.Tensor:
        x = self.embedding(source) + sinusoidal_positions(source.size(1), self.cfg.d_model, source.device)
        return self.encoder(x)

    def decode(self, target: torch.Tensor, memory: torch.Tensor) -> torch.Tensor:
        y = self.embedding(target) + sinusoidal_positions(target.size(1), self.cfg.d_model, target.device)
        y = self.decoder(y, memory, tgt_mask=causal_mask(target.size(1), target.device))
        source_summary = memory.mean(dim=1, keepdim=True).expand(-1, y.size(1), -1)
        y = self.fusion(y, source_summary)
        return self.head(self.norm(y))

    def loss(self, batch: torch.Tensor, bos: int) -> torch.Tensor:
        split = batch.size(1) // 2
        source, target = batch[:, :split], batch[:, split:]
        decoder_input = torch.cat((torch.full_like(target[:, :1], bos), target[:, :-1]), dim=1)
        return F.cross_entropy(self.decode(decoder_input, self.encode(source)).flatten(0, 1), target.flatten())

    def next_logits(self, source: torch.Tensor, generated: torch.Tensor, bos: int) -> torch.Tensor:
        memory = self.encode(source[:, -self.cfg.seq_len :])
        target = generated[:, -self.cfg.seq_len :]
        if target.size(1) == 0:
            target = torch.full((source.size(0), 1), bos, dtype=torch.long, device=source.device)
        return self.decode(target, memory)[:, -1]


class HybridLM(EncoderDecoderLM):
    """Bidirectional recurrent encoder plus causal Transformer decoder."""

    def __init__(self, cfg: RunConfig, recurrent_layers: int = 5, decoder_layers: int = 11):
        nn.Module.__init__(self)
        self.cfg = cfg
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.recurrent = nn.GRU(
            cfg.d_model, cfg.d_model // 2, recurrent_layers,
            batch_first=True, bidirectional=True, dropout=0.0,
        )
        decoder = nn.TransformerDecoderLayer(
            cfg.d_model, cfg.n_heads, 4 * cfg.d_model, batch_first=True,
            norm_first=True, activation="gelu", dropout=0.0,
        )
        self.decoder = nn.TransformerDecoder(decoder, decoder_layers)
        self.fusion = GatedContextFusion(cfg.d_model)
        self.norm = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.embedding.weight

    def encode(self, source: torch.Tensor) -> torch.Tensor:
        x = self.embedding(source) + sinusoidal_positions(source.size(1), self.cfg.d_model, source.device)
        return self.recurrent(x)[0]


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight


def rope(x: torch.Tensor) -> torch.Tensor:
    # x: batch, heads, time, head_dim
    length, dim = x.size(-2), x.size(-1)
    position = torch.arange(length, device=x.device).float()
    frequency = torch.exp(torch.arange(0, dim, 2, device=x.device).float() * (-math.log(10000.0) / dim))
    angle = torch.outer(position, frequency)
    cos, sin = angle.cos()[None, None], angle.sin()[None, None]
    even, odd = x[..., 0::2], x[..., 1::2]
    return torch.stack((even * cos - odd * sin, even * sin + odd * cos), dim=-1).flatten(-2)


class GroupedQueryAttention(nn.Module):
    def __init__(self, dim: int, heads: int, kv_heads: int = 2, window: int = 64):
        super().__init__()
        self.heads, self.kv_heads, self.head_dim, self.window = heads, kv_heads, dim // heads, window
        self.q = nn.Linear(dim, heads * self.head_dim, bias=False)
        self.k = nn.Linear(dim, kv_heads * self.head_dim, bias=False)
        self.v = nn.Linear(dim, kv_heads * self.head_dim, bias=False)
        self.out = nn.Linear(dim, dim, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, length, _ = x.shape
        q = rope(self.q(x).view(batch, length, self.heads, self.head_dim).transpose(1, 2))
        k = rope(self.k(x).view(batch, length, self.kv_heads, self.head_dim).transpose(1, 2))
        v = self.v(x).view(batch, length, self.kv_heads, self.head_dim).transpose(1, 2)
        repeat = self.heads // self.kv_heads
        k, v = k.repeat_interleave(repeat, 1), v.repeat_interleave(repeat, 1)
        positions = torch.arange(length, device=x.device)
        distance = positions[:, None] - positions[None, :]
        if getattr(self, "bidirectional", False):
            allowed = distance.abs() < self.window
        else:
            allowed = (distance >= 0) & (distance < self.window)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(~allowed, torch.finfo(scores.dtype).min)
        output = torch.matmul(scores.softmax(-1), v).transpose(1, 2).contiguous().view(batch, length, -1)
        return self.out(output)


class EfficientBlock(nn.Module):
    def __init__(self, cfg: RunConfig):
        super().__init__()
        hidden = int(8 * cfg.d_model / 3)
        hidden += (-hidden) % 64
        self.norm1, self.norm2 = RMSNorm(cfg.d_model), RMSNorm(cfg.d_model)
        self.attention = GroupedQueryAttention(cfg.d_model, cfg.n_heads)
        self.gate = nn.Linear(cfg.d_model, hidden, bias=False)
        self.up = nn.Linear(cfg.d_model, hidden, bias=False)
        self.down = nn.Linear(hidden, cfg.d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attention(self.norm1(x))
        y = self.norm2(x)
        return x + self.down(F.silu(self.gate(y)) * self.up(y))


class EfficientLM(nn.Module):
    def __init__(self, cfg: RunConfig, layers: int = 19):
        super().__init__()
        self.cfg = cfg
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList(EfficientBlock(cfg) for _ in range(layers))
        self.norm = RMSNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.embedding.weight

    def logits(self, ids: torch.Tensor) -> torch.Tensor:
        x = self.embedding(ids)
        for block in self.blocks:
            x = block(x)
        return self.head(self.norm(x))

    def loss(self, batch: torch.Tensor, bos: int) -> torch.Tensor:
        del bos
        return F.cross_entropy(self.logits(batch[:, :-1]).flatten(0, 1), batch[:, 1:].flatten())

    def next_logits(self, source: torch.Tensor, generated: torch.Tensor, bos: int) -> torch.Tensor:
        del bos
        context = torch.cat((source, generated), dim=1)[:, -self.cfg.seq_len :]
        return self.logits(context)[:, -1]


def build_model(name: str, cfg: RunConfig) -> nn.Module:
    builders = {
        "decoder": DecoderOnlyLM,
        "encoder_decoder": EncoderDecoderLM,
        "hybrid": HybridLM,
        "efficient": EfficientLM,
    }
    return builders[name](cfg)


def parameter_count(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters())


def evaluate_loss(model: nn.Module, loader, bos: int, device: torch.device, batches: int = 20) -> float:
    model.eval()
    losses = []
    with torch.inference_mode():
        for index, batch in enumerate(loader):
            if index >= batches:
                break
            batch = batch.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
                losses.append(float(model.loss(batch, bos)))
    model.train()
    return sum(losses) / len(losses)


def train_model(
    model: nn.Module,
    loader,
    validation_loader,
    cfg: RunConfig,
    bos: int,
    device: torch.device,
    recovery_path: Path,
    start_step: int = 0,
    prior_losses: list[float] | None = None,
) -> tuple[list[float], float]:
    model.to(device).train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    warmup = max(1, cfg.steps // 20)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lambda step: (step + 1) / warmup
        if step < warmup
        else 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * (step - warmup) / max(1, cfg.steps - warmup))),
    )
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    losses, iterator = list(prior_losses or []), iter(loader)
    optimizer.zero_grad(set_to_none=True)
    for step in range(start_step, cfg.steps):
        running = 0.0
        for _ in range(cfg.grad_accum):
            try:
                batch = next(iterator)
            except StopIteration:
                iterator = iter(loader)
                batch = next(iterator)
            batch = batch.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
                loss = model.loss(batch, bos) / cfg.grad_accum
            scaler.scale(loss).backward()
            running += float(loss.detach())
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        losses.append(running)
        if step == 0 or (step + 1) % 10 == 0:
            print(f"step {step + 1:>3}/{cfg.steps}: loss={running:.4f}")
        if (step + 1) % 25 == 0 or step + 1 == cfg.steps:
            temporary = recovery_path.with_suffix(".tmp")
            torch.save(
                {
                    "model": model.state_dict(),
                    "model_name": recovery_path.stem.removesuffix("_recovery"),
                    "completed_steps": step + 1,
                    "config": asdict(cfg),
                    "train_loss": losses,
                },
                temporary,
            )
            os.replace(temporary, recovery_path)
            print(f"recovery checkpoint saved: {recovery_path.name}")
    return losses, evaluate_loss(model, validation_loader, bos, device)


def sample_next(logits: torch.Tensor, temperature: float = 0.2, top_k: int = 20) -> torch.Tensor:
    logits = logits / temperature
    values, indices = torch.topk(logits, min(top_k, logits.size(-1)), dim=-1)
    sampled = torch.multinomial(values.softmax(-1), 1)
    return indices.gather(1, sampled)


@torch.inference_mode()
def generate_predictions(model, tokenizer, test_data, cfg: RunConfig, device: torch.device) -> dict:
    model.eval()
    pad, bos, eos = (tokenizer.token_to_id(token) for token in SPECIAL[:3])
    items, predictions = list(test_data.items()), {}
    for start in range(0, len(items), cfg.generation_batch_size):
        batch = items[start : start + cfg.generation_batch_size]
        encoded = [tokenizer.encode(record["text"]).ids[-cfg.seq_len :] for _, record in batch]
        width = max(map(len, encoded))
        source = torch.full((len(encoded), width), pad, dtype=torch.long, device=device)
        for row, ids in enumerate(encoded):
            source[row, -len(ids) :] = torch.tensor(ids, device=device)
        generated = torch.empty((len(encoded), 0), dtype=torch.long, device=device)
        finished_text = [None] * len(encoded)
        # Encoder-decoder models reuse their source representation at every step.
        memory = model.encode(source) if isinstance(model, EncoderDecoderLM) else None
        for _ in range(cfg.max_generation_tokens):
            if memory is None:
                next_logits = model.next_logits(source, generated, bos)
            else:
                target = generated[:, -cfg.seq_len :]
                if target.size(1) == 0:
                    target = torch.full((source.size(0), 1), bos, dtype=torch.long, device=device)
                next_logits = model.decode(target, memory)[:, -1]
            next_id = sample_next(next_logits)
            next_id[next_id.eq(eos)] = bos
            generated = torch.cat((generated, next_id), dim=1)
            if generated.size(1) % 16 == 0:
                for row in range(len(encoded)):
                    words = tokenizer.decode(generated[row].tolist(), skip_special_tokens=True).split()
                    if len(words) >= cfg.target_words:
                        finished_text[row] = " ".join(words[: cfg.target_words])
                if all(text is not None for text in finished_text):
                    break
        for row, (example_id, _) in enumerate(batch):
            if finished_text[row] is None:
                words = tokenizer.decode(generated[row].tolist(), skip_special_tokens=True).split()
                if not words:
                    words = ["the"]
                words = (words * math.ceil(cfg.target_words / len(words)))[: cfg.target_words]
                finished_text[row] = " ".join(words)
            predictions[str(example_id)] = {"text": finished_text[row]}
        print(f"generated {min(start + len(batch), len(items))}/{len(items)}")
    return predictions


def validate_and_zip(predictions: dict, test_data: dict, output_dir: Path, name: str) -> Path:
    expected, actual = {str(key) for key in test_data}, set(predictions)
    if expected != actual:
        raise ValueError(f"ID mismatch: missing={expected - actual}, extra={actual - expected}")
    for example_id, record in predictions.items():
        if set(record) != {"text"} or len(record["text"].split()) != 128:
            raise ValueError(f"Invalid prediction for {example_id}")
    folder = output_dir / name
    folder.mkdir(parents=True, exist_ok=True)
    prediction_path = folder / "prediction.json"
    prediction_path.write_text(json.dumps(predictions, ensure_ascii=False, indent=2), encoding="utf-8")
    zip_path = output_dir / f"submission_{name}.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
        archive.write(prediction_path, arcname="prediction.json")
    with zipfile.ZipFile(zip_path) as archive:
        if archive.namelist() != ["prediction.json"]:
            raise ValueError(f"Bad archive layout: {archive.namelist()}")
    return zip_path


def run(args) -> None:
    seed_everything()
    cfg = RunConfig(
        steps=args.steps,
        train_chars=args.train_chars,
        train_tokens=args.train_tokens,
        seq_len=args.seq_len,
        batch_size=args.batch_size,
        grad_accum=args.grad_accum,
    )
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    tokenizer_path = output_dir / "phase1_bpe.json"
    if args.generate_only:
        if args.model == "all" or not args.resume_checkpoint:
            raise ValueError("--generate-only requires one --model and --resume-checkpoint")
        source_tokenizer = Path(args.tokenizer_path) if args.tokenizer_path else tokenizer_path
        tokenizer = load_tokenizer(source_tokenizer)
        cfg.vocab_size = tokenizer.get_vocab_size()
        model = build_model(args.model, cfg)
        saved = torch.load(args.resume_checkpoint, map_location="cpu")
        model.load_state_dict(saved["model"])
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model.to(device)
        test_data = load_test_data()
        predictions = generate_predictions(model, tokenizer, test_data, cfg, device)
        zip_path = validate_and_zip(predictions, test_data, output_dir, args.model)
        print(f"created {zip_path}")
        return
    texts = collect_training_text(cfg)
    random.Random(SEED).shuffle(texts)
    if args.validation_prompts and args.validation_truth:
        validation_prompts = json.loads(Path(args.validation_prompts).read_text(encoding="utf-8"))
        validation_truth = json.loads(Path(args.validation_truth).read_text(encoding="utf-8"))
        texts = exclude_generation_validation(texts, validation_prompts, validation_truth)
    elif args.validation_prompts or args.validation_truth:
        raise ValueError("Provide both --validation-prompts and --validation-truth")
    else:
        texts, validation_prompts, validation_truth = make_generation_validation(texts)
    validation_prompts_path = output_dir / "validation_prompts.json"
    validation_truth_path = output_dir / "validation_truth.json"
    validation_prompts_path.write_text(
        json.dumps(validation_prompts, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    validation_truth_path.write_text(
        json.dumps(validation_truth, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    if args.tokenizer_path:
        tokenizer = load_tokenizer(Path(args.tokenizer_path))
        tokenizer.save(str(tokenizer_path))
    else:
        tokenizer_texts, used = [], 0
        for text in texts:
            tokenizer_texts.append(text)
            used += len(text)
            if used >= cfg.tokenizer_chars:
                break
        tokenizer = train_tokenizer(tokenizer_texts, tokenizer_path, cfg.vocab_size)
    cfg.vocab_size = tokenizer.get_vocab_size()
    token_ids = tokenize_training_text(texts, tokenizer, cfg)
    split = int(0.95 * len(token_ids))
    train_ids, validation_ids = token_ids[:split], token_ids[split:]
    samples = cfg.steps * cfg.grad_accum * cfg.batch_size
    dataset = TokenBlocks(train_ids, cfg.seq_len + 1, samples, SEED)
    loader = torch.utils.data.DataLoader(dataset, batch_size=cfg.batch_size, shuffle=False, pin_memory=True)
    validation = TokenBlocks(validation_ids, cfg.seq_len + 1, 20 * cfg.batch_size, SEED + 1)
    validation_loader = torch.utils.data.DataLoader(
        validation, batch_size=cfg.batch_size, shuffle=False, pin_memory=True
    )
    test_data = load_test_data()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    bos = tokenizer.token_to_id("<bos>")
    manifest_path = output_dir / "manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else {}
    if args.models:
        selected = tuple(name.strip() for name in args.models.split(",") if name.strip())
        invalid = set(selected) - set(MODEL_NAMES)
        if invalid or not selected:
            raise ValueError(f"Invalid --models selection: {sorted(invalid)}")
    else:
        selected = MODEL_NAMES if args.model == "all" else (args.model,)
    for name in selected:
        seed_everything(SEED + MODEL_NAMES.index(name))
        print(f"\n{'=' * 20} {name} {'=' * 20}")
        model = build_model(name, cfg)
        previous_steps = 0
        recovery_path = output_dir / f"{name}_recovery.pt"
        recovery_step = 0
        recovery_losses = []
        if args.resume_checkpoint:
            if len(selected) != 1:
                raise ValueError("--resume-checkpoint requires one selected model")
            saved = torch.load(args.resume_checkpoint, map_location="cpu")
            saved_name = saved.get("model_name")
            if saved_name and saved_name != name:
                raise ValueError(f"Checkpoint model is {saved_name}, not {name}")
            model.load_state_dict(saved["model"])
            previous_steps = int(saved.get("total_steps", saved.get("config", {}).get("steps", 0)))
            print(f"resumed {name} after {previous_steps:,} prior steps")
        elif recovery_path.exists():
            saved = torch.load(recovery_path, map_location="cpu")
            model.load_state_dict(saved["model"])
            recovery_step = int(saved["completed_steps"])
            recovery_losses = list(saved.get("train_loss", []))
            print(f"automatically resumed {name} at step {recovery_step}/{cfg.steps}")
        parameters = parameter_count(model)
        if not 50_000_000 <= parameters <= 300_000_000:
            raise ValueError(f"{name} has {parameters:,} parameters; must be 50M-300M")
        print(f"parameters: {parameters:,}")
        started = time.perf_counter()
        losses, validation_loss = train_model(
            model,
            loader,
            validation_loader,
            cfg,
            bos,
            device,
            recovery_path,
            recovery_step,
            recovery_losses,
        )
        elapsed = time.perf_counter() - started
        log_path = output_dir / f"training_log_{name}.json"
        log_path.write_text(
            json.dumps(
                {
                    "config": asdict(cfg),
                    "train_loss": losses,
                    "validation_loss": validation_loss,
                    "training_seconds": elapsed,
                },
                indent=2,
            ),
            encoding="utf-8",
        )
        checkpoint = output_dir / f"{name}.pt"
        torch.save(
            {
                "model": model.state_dict(),
                "config": asdict(cfg),
                "parameters": parameters,
                "model_name": name,
                "total_steps": previous_steps + cfg.steps,
                "train_loss": losses,
                "validation_loss": validation_loss,
            },
            checkpoint,
        )
        validation_predictions = generate_predictions(
            model, tokenizer, validation_prompts, cfg, device
        )
        validation_prediction_path = output_dir / f"validation_prediction_{name}.json"
        validation_prediction_path.write_text(
            json.dumps(validation_predictions, ensure_ascii=False, indent=2), encoding="utf-8"
        )
        predictions = generate_predictions(model, tokenizer, test_data, cfg, device)
        zip_path = validate_and_zip(predictions, test_data, output_dir, name)
        manifest[name] = {
            "category": {
                "decoder": "decoder-only",
                "encoder_decoder": "encoder-decoder",
                "hybrid": "encoder/hybrid",
                "efficient": "efficient/lightweight",
            }[name],
            "parameters": parameters,
            "steps_this_run": cfg.steps,
            "total_steps": previous_steps + cfg.steps,
            "final_loss": losses[-1],
            "validation_loss": validation_loss,
            "validation_perplexity": math.exp(min(validation_loss, 20.0)),
            "training_seconds": elapsed,
            "device": str(device),
            "checkpoint": str(checkpoint),
            "training_log": str(log_path),
            "validation_truth": str(validation_truth_path),
            "validation_prompts": str(validation_prompts_path),
            "validation_prediction": str(validation_prediction_path),
            "submission": str(zip_path),
        }
        manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
        recovery_path.unlink(missing_ok=True)
        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()
    print(json.dumps(manifest, indent=2))


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", choices=("all",) + MODEL_NAMES, default="all")
    parser.add_argument("--models", help="Comma-separated subset for multi-GPU orchestration")
    parser.add_argument("--steps", type=int, default=400)
    parser.add_argument("--train-chars", type=int, default=40_000_000)
    parser.add_argument("--train-tokens", type=int, default=12_000_000)
    parser.add_argument("--seq-len", type=int, default=256)
    parser.add_argument("--batch-size", type=int, default=2)
    parser.add_argument("--grad-accum", type=int, default=4)
    parser.add_argument("--tokenizer-path")
    parser.add_argument("--resume-checkpoint")
    parser.add_argument("--generate-only", action="store_true")
    parser.add_argument("--validation-prompts")
    parser.add_argument("--validation-truth")
    parser.add_argument("--output-dir", default="/kaggle/working/phase1")
    return parser.parse_args()


if __name__ == "__main__":
    run(parse_args())


In [ ]:
%%writefile /kaggle/working/phase1_foundation.py
"""Train the Phase-1 flagship legal encoder-decoder model from scratch.

This file deliberately reuses only the proven corpus/test download helpers from
phase1_pipeline.py.  The tokenizer, model, objective, checkpointing, validation,
generation, and submission path are implemented here.
"""

from __future__ import annotations

import argparse
import bisect
import contextlib
import json
import math
import os
import random
import time
from array import array
from dataclasses import asdict, dataclass
from datetime import timedelta
from pathlib import Path

import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.checkpoint import checkpoint
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers

import phase1_pipeline as legacy


SEED = 781
SPECIAL = ("<pad>", "<bos>", "<eos>", "<unk>", "<mask>")


def init_distributed():
    distributed = "RANK" in os.environ and "WORLD_SIZE" in os.environ
    if not distributed:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        return False, 0, 1, 0, device
    if not torch.cuda.is_available():
        raise RuntimeError("Distributed training requires CUDA")
    rank = int(os.environ["RANK"])
    world_size = int(os.environ["WORLD_SIZE"])
    local_rank = int(os.environ["LOCAL_RANK"])
    if world_size > torch.cuda.device_count():
        raise RuntimeError(f"Requested {world_size} ranks but found {torch.cuda.device_count()} GPUs")
    torch.cuda.set_device(local_rank)
    dist.init_process_group("nccl", timeout=timedelta(minutes=30))
    return True, rank, world_size, local_rank, torch.device("cuda", local_rank)


def distributed_barrier(distributed):
    if distributed:
        dist.barrier()


def unwrap_model(model):
    return model.module if isinstance(model, DDP) else model


@dataclass
class Config:
    vocab_size: int = 24_000
    d_model: int = 640
    heads: int = 10
    encoder_layers: int = 12
    decoder_layers: int = 8
    d_ff: int = 1_728
    source_len: int = 1_024
    target_len: int = 256
    global_block: int = 32
    dropout: float = 0.1
    steps: int = 60_000
    batch_size: int = 1
    grad_accum: int = 8
    learning_rate: float = 3e-4
    weight_decay: float = 0.1
    warmup_fraction: float = 0.02
    train_chars: int = 500_000_000
    token_pool: int = 100_000_000
    tokenizer_chars: int = 80_000_000
    validation_examples: int = 256
    validation_batches: int = 24
    checkpoint_every: int = 100
    mlm_probability: float = 0.25
    mlm_mask_fraction: float = 0.15
    mlm_weight: float = 0.35
    generation_batch_size: int = 8
    max_generation_tokens: int = 384
    target_words: int = 128
    use_global_memory: bool = True


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def train_tokenizer(texts: list[str], path: Path, cfg: Config) -> Tokenizer:
    tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=cfg.vocab_size,
        min_frequency=2,
        special_tokens=list(SPECIAL),
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
        show_progress=True,
    )
    used = 0

    def iterator():
        nonlocal used
        for text in texts:
            if used >= cfg.tokenizer_chars:
                break
            used += len(text)
            yield text

    tokenizer.train_from_iterator(iterator(), trainer=trainer)
    tokenizer.save(str(path))
    return tokenizer


def make_validation(texts: list[str], count: int) -> tuple[list[str], list[str], dict, dict]:
    rng = random.Random(SEED + 11)
    order = list(range(len(texts)))
    rng.shuffle(order)
    used: set[int] = set()
    prompts, truth = {}, {}
    for index in order:
        words = texts[index].split()
        if len(words) < 640:
            continue
        start = rng.randrange(len(words) - 640 + 1)
        key = f"validation_{len(prompts):04d}"
        prompts[key] = {"text": " ".join(words[start : start + 512])}
        truth[key] = {"text": " ".join(words[start + 512 : start + 640])}
        used.add(index)
        if len(prompts) == count:
            break
    if len(prompts) < count:
        raise RuntimeError(f"Only {len(prompts)} documents could supply validation examples")
    training = [text for i, text in enumerate(texts) if i not in used]
    validation = [text for i, text in enumerate(texts) if i in used]
    return training, validation, prompts, truth


def tokenize_documents(
    texts: list[str],
    tokenizer: Tokenizer,
    cfg: Config,
    source_len: int | None = None,
    target_len: int | None = None,
):
    eos = tokenizer.token_to_id("<eos>")
    flat, spans = array("I"), []
    block = (source_len or cfg.source_len) + (target_len or cfg.target_len)
    for text in texts:
        ids = tokenizer.encode(text).ids
        if len(ids) < block:
            continue
        remaining = cfg.token_pool - len(flat)
        if remaining <= 0:
            break
        ids = ids[:remaining]
        if len(ids) < block:
            break
        start = len(flat)
        flat.extend(ids)
        flat.append(eos)
        spans.append((start, start + len(ids)))
    if not spans:
        raise RuntimeError("No documents were long enough for the configured source/target lengths")
    return flat, spans


class ContinuationDataset(torch.utils.data.Dataset):
    def __init__(
        self,
        tokens,
        spans,
        source_len,
        target_len,
        samples,
        seed,
        sample_offset=0,
        rank=0,
        world_size=1,
        samples_per_step=1,
    ):
        self.tokens, self.spans = tokens, spans
        self.source_len, self.target_len = source_len, target_len
        self.samples, self.seed, self.sample_offset = samples, seed, sample_offset
        self.rank, self.world_size = rank, world_size
        self.samples_per_step = samples_per_step
        block = source_len + target_len
        self.cumulative, total = [], 0
        for start, end in spans:
            total += end - start - block + 1
            self.cumulative.append(total)
        self.total = total

    def __len__(self):
        return self.samples

    def __getitem__(self, index):
        step, within_step = divmod(index, self.samples_per_step)
        global_index = (
            self.sample_offset
            + step * self.samples_per_step * self.world_size
            + self.rank * self.samples_per_step
            + within_step
        )
        pick = random.Random(self.seed + global_index).randrange(self.total)
        doc = bisect.bisect_right(self.cumulative, pick)
        previous = self.cumulative[doc - 1] if doc else 0
        start = self.spans[doc][0] + pick - previous
        stop = start + self.source_len + self.target_len
        values = torch.tensor(self.tokens[start:stop], dtype=torch.long)
        return values[: self.source_len], values[self.source_len :]


class RMSNorm(nn.Module):
    def __init__(self, width: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(width))
        self.eps = eps

    def forward(self, x):
        scale = x.float().pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt().to(x.dtype)
        return x * scale * self.weight


def rotate_half(x):
    left, right = x.chunk(2, dim=-1)
    return torch.cat((-right, left), dim=-1)


def apply_rope(q, k):
    length, width = q.size(-2), q.size(-1)
    positions = torch.arange(length, device=q.device, dtype=torch.float32)
    inverse = 1.0 / (10_000 ** (torch.arange(0, width, 2, device=q.device).float() / width))
    angles = torch.outer(positions, inverse)
    angles = torch.cat((angles, angles), dim=-1)[None, None]
    cos, sin = angles.cos().to(q.dtype), angles.sin().to(q.dtype)
    return q * cos + rotate_half(q) * sin, k * cos + rotate_half(k) * sin


class Attention(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.heads, self.head_dim = cfg.heads, cfg.d_model // cfg.heads
        self.q = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.k = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.v = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.out = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.dropout = cfg.dropout

    def forward(self, query, memory=None, key_mask=None, causal=False, rope=False):
        memory = query if memory is None else memory
        batch, query_len, _ = query.shape
        key_len = memory.size(1)

        def split(projection, values, length):
            return projection(values).view(batch, length, self.heads, self.head_dim).transpose(1, 2)

        q = split(self.q, query, query_len)
        k = split(self.k, memory, key_len)
        v = split(self.v, memory, key_len)
        if rope:
            q, k = apply_rope(q, k)
        mask = None if key_mask is None else key_mask[:, None, None, :]
        if causal and mask is not None:
            causal_mask = torch.ones(query_len, key_len, device=query.device, dtype=torch.bool).tril()
            mask = mask & causal_mask[None, None]
            causal = False
        result = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=causal,
        )
        result = result.transpose(1, 2).contiguous().view(batch, query_len, -1)
        return self.out(result)


class SwiGLU(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.value = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.out = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)

    def forward(self, x):
        return self.out(F.silu(self.gate(x)) * self.value(x))


class EncoderBlock(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.attn, self.ff = Attention(cfg), SwiGLU(cfg)
        self.attn_norm, self.ff_norm = RMSNorm(cfg.d_model), RMSNorm(cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, mask=None):
        normalized = self.attn_norm(x)
        x = x + self.dropout(self.attn(normalized, key_mask=mask, rope=True))
        return x + self.dropout(self.ff(self.ff_norm(x)))


class DecoderBlock(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.self_attn, self.cross_attn, self.ff = Attention(cfg), Attention(cfg), SwiGLU(cfg)
        self.self_norm = RMSNorm(cfg.d_model)
        self.cross_norm = RMSNorm(cfg.d_model)
        self.ff_norm = RMSNorm(cfg.d_model)
        self.memory_gate = nn.Linear(cfg.d_model, 1)
        self.dropout = nn.Dropout(cfg.dropout)
        self.use_global_memory = cfg.use_global_memory

    def forward(self, x, memory, global_memory, memory_mask=None, global_mask=None):
        normalized = self.self_norm(x)
        x = x + self.dropout(self.self_attn(normalized, causal=True, rope=True))
        normalized = self.cross_norm(x)
        token_context = self.cross_attn(normalized, memory, key_mask=memory_mask)
        if self.use_global_memory:
            global_context = self.cross_attn(normalized, global_memory, key_mask=global_mask)
            gate = torch.sigmoid(self.memory_gate(normalized))
            token_context = gate * token_context + (1.0 - gate) * global_context
        x = x + self.dropout(token_context)
        return x + self.dropout(self.ff(self.ff_norm(x)))


class LegalEncoderDecoder(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.encoder = nn.ModuleList(EncoderBlock(cfg) for _ in range(cfg.encoder_layers))
        self.decoder = nn.ModuleList(DecoderBlock(cfg) for _ in range(cfg.decoder_layers))
        self.encoder_norm = RMSNorm(cfg.d_model)
        self.decoder_norm = RMSNorm(cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)
        self.apply(self._init)

    def _init(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, std=0.02)

    def encode(self, source, source_mask=None):
        x = self.dropout(self.embedding(source))
        for layer in self.encoder:
            if self.training:
                x = checkpoint(layer, x, source_mask, use_reentrant=False)
            else:
                x = layer(x, source_mask)
        return self.encoder_norm(x)

    def pool_memory(self, memory, mask=None):
        block = self.cfg.global_block
        batch, length, width = memory.shape
        padding = (-length) % block
        if padding:
            memory = F.pad(memory, (0, 0, 0, padding))
            if mask is not None:
                mask = F.pad(mask, (0, padding), value=False)
        memory = memory.view(batch, -1, block, width)
        if mask is None:
            return memory.mean(2), None
        mask = mask.view(batch, -1, block)
        counts = mask.sum(2, keepdim=True).clamp_min(1)
        pooled = (memory * mask[..., None]).sum(2) / counts
        return pooled, mask.any(2)

    def decode(self, decoder_input, memory, source_mask=None):
        x = self.dropout(self.embedding(decoder_input))
        global_memory, global_mask = self.pool_memory(memory, source_mask)
        for layer in self.decoder:
            if self.training:
                x = checkpoint(
                    layer, x, memory, global_memory, source_mask, global_mask, use_reentrant=False
                )
            else:
                x = layer(x, memory, global_memory, source_mask, global_mask)
        return self.decoder_norm(x)

    def logits(self, hidden):
        return F.linear(hidden, self.embedding.weight)

    def forward(self, source, decoder_input, source_mask=None):
        memory = self.encode(source, source_mask)
        return self.logits(self.decode(decoder_input, memory, source_mask)), memory


def parameter_count(model):
    return sum(parameter.numel() for parameter in model.parameters())


def corrupt_source(source, mask_id, pad_id, fraction):
    selected = torch.rand(source.shape, device=source.device) < fraction
    selected &= source.ne(pad_id)
    corrupted = source.clone()
    corrupted[selected] = mask_id
    return corrupted, selected


def cosine_multiplier(step, total, warmup):
    if step < warmup:
        return (step + 1) / max(1, warmup)
    progress = (step - warmup) / max(1, total - warmup)
    return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress))


def atomic_save(payload, path: Path):
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, path)


def evaluate_loss(model, loader, bos, device, batches):
    model.eval()
    losses = []
    with torch.inference_mode():
        for number, (source, target) in enumerate(loader):
            if number == batches:
                break
            source, target = source.to(device), target.to(device)
            decoder_input = torch.cat((torch.full_like(target[:, :1], bos), target[:, :-1]), 1)
            logits, _ = model(source, decoder_input)
            losses.append(F.cross_entropy(logits.flatten(0, 1), target.flatten()).item())
    model.train()
    return sum(losses) / len(losses)


def train(
    model,
    loader,
    validation_loader,
    tokenizer,
    cfg,
    output_dir,
    device,
    resume=None,
    run_steps=None,
    distributed=False,
    rank=0,
    world_size=1,
    max_hours=None,
    generation_reserve_hours=0.0,
    session_started=None,
    finalize_on_time_limit=False,
):
    if cfg.grad_accum % world_size:
        raise ValueError("--grad-accum must be divisible by the number of DDP processes")
    local_grad_accum = cfg.grad_accum // world_size
    raw_model = unwrap_model(model)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg.learning_rate, betas=(0.9, 0.95), weight_decay=cfg.weight_decay
    )
    warmup = int(cfg.steps * cfg.warmup_fraction)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda step: cosine_multiplier(step, cfg.steps, warmup)
    )
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    amp_dtype = (
        torch.bfloat16
        if device.type == "cuda" and torch.cuda.is_bf16_supported()
        else torch.float16
    )
    start_step, losses = 0, []
    if resume:
        saved = torch.load(resume, map_location="cpu", weights_only=False)
        state = saved["model"]
        if state and all(key.startswith("module.") for key in state):
            state = {key.removeprefix("module."): value for key, value in state.items()}
        raw_model.load_state_dict(state)
        if "optimizer" in saved:
            optimizer.load_state_dict(saved["optimizer"])
            scheduler.load_state_dict(saved["scheduler"])
            scaler.load_state_dict(saved["scaler"])
        start_step = int(saved.get("completed_steps", saved.get("total_steps", 0)))
        losses = list(saved.get("train_loss", []))
        rng_states = saved.get("rng_states")
        if rng_states and rank < len(rng_states):
            rng = rng_states[rank]
            random.setstate(rng["python"])
            torch.set_rng_state(rng["torch"])
            if device.type == "cuda" and rng.get("cuda") is not None:
                torch.cuda.set_rng_state(rng["cuda"], device)
        else:
            seed_everything(SEED + rank + start_step)
        if rank == 0:
            print(f"resumed at step {start_step}/{cfg.steps}")
    else:
        seed_everything(SEED + rank)
    bos = tokenizer.token_to_id("<bos>")
    pad = tokenizer.token_to_id("<pad>")
    mask_id = tokenizer.token_to_id("<mask>")
    iterator = iter(loader)
    recovery = output_dir / "foundation_recovery.pt"
    model.train()
    optimizer.zero_grad(set_to_none=True)
    end_step = min(cfg.steps, start_step + run_steps) if run_steps else cfg.steps
    if max_hours and not 0 <= generation_reserve_hours < max_hours:
        raise ValueError("--generation-reserve-hours must be non-negative and below --max-hours")
    training_hours = max_hours - generation_reserve_hours if max_hours else None
    deadline = (
        (session_started if session_started is not None else time.perf_counter())
        + training_hours * 3600
        if training_hours
        else None
    )
    if rank == 0 and max_hours:
        print(
            f"total runtime budget: {max_hours:.2f}h; training/data budget: "
            f"{training_hours:.2f}h; generation reserve: {generation_reserve_hours:.2f}h",
            flush=True,
        )
    completed = start_step
    stopped_by_time = False

    def save_recovery():
        local_rng = {
            "python": random.getstate(),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state(device) if device.type == "cuda" else None,
        }
        if distributed:
            rng_states = [None] * world_size
            dist.all_gather_object(rng_states, local_rng)
        else:
            rng_states = [local_rng]
        if rank == 0:
            atomic_save(
                {
                    "model": raw_model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(),
                    "scaler": scaler.state_dict(),
                    "completed_steps": completed,
                    "config": asdict(cfg),
                    "parameters": parameter_count(raw_model),
                    "train_loss": losses,
                    "rng_states": rng_states,
                    "world_size": world_size,
                },
                recovery,
            )
            print(f"checkpoint saved: {recovery}", flush=True)
        distributed_barrier(distributed)

    for step in range(start_step, end_step):
        running = 0.0
        for micro_step in range(local_grad_accum):
            try:
                source, target = next(iterator)
            except StopIteration:
                iterator = iter(loader)
                source, target = next(iterator)
            source, target = source.to(device), target.to(device)
            decoder_input = torch.cat((torch.full_like(target[:, :1], bos), target[:, :-1]), 1)
            use_mlm = random.random() < cfg.mlm_probability
            model_source, selected = (
                corrupt_source(source, mask_id, pad, cfg.mlm_mask_fraction)
                if use_mlm
                else (source, None)
            )
            synchronize = micro_step == local_grad_accum - 1
            sync_context = (
                contextlib.nullcontext()
                if synchronize or not distributed
                else model.no_sync()
            )
            with sync_context:
                with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=device.type == "cuda"):
                    logits, memory = model(model_source, decoder_input)
                    loss = F.cross_entropy(logits.flatten(0, 1), target.flatten())
                    if selected is not None and selected.any():
                        encoder_logits = raw_model.logits(memory[selected])
                        loss = loss + cfg.mlm_weight * F.cross_entropy(
                            encoder_logits, source[selected]
                        )
                    loss = loss / local_grad_accum
                scaler.scale(loss).backward()
            running += float(loss.detach())
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        if distributed:
            reduced = torch.tensor(running, dtype=torch.float32, device=device)
            dist.all_reduce(reduced, op=dist.ReduceOp.SUM)
            running = reduced.item() / world_size
        losses.append(running)
        completed = step + 1
        if rank == 0 and (completed == 1 or completed % 10 == 0):
            tokens = completed * cfg.batch_size * cfg.grad_accum * (cfg.source_len + cfg.target_len)
            elapsed = time.perf_counter() - (session_started or time.perf_counter())
            rate = elapsed / max(1, completed - start_step)
            print(
                f"step {completed}/{cfg.steps}: loss={running:.4f}, tokens={tokens:,}, "
                f"elapsed={elapsed / 3600:.2f}h, seconds/step={rate:.2f}",
                flush=True,
            )
        time_limit = bool(deadline and time.perf_counter() >= deadline)
        if distributed:
            stop = torch.tensor(int(time_limit), device=device)
            dist.broadcast(stop, src=0)
            time_limit = bool(stop.item())
        if completed % cfg.checkpoint_every == 0 or completed == end_step or time_limit:
            save_recovery()
        if time_limit:
            stopped_by_time = True
            break
    should_finalize = completed >= cfg.steps or (stopped_by_time and finalize_on_time_limit)
    if not should_finalize:
        if rank == 0:
            reason = "time limit" if stopped_by_time else "chunk limit"
            print(f"{reason} reached at {completed}/{cfg.steps}; resume from {recovery}")
        return recovery, None, False, completed, "time_limit" if stopped_by_time else "chunk_limit"
    distributed_barrier(distributed)
    validation_loss = None
    if rank == 0:
        validation_loss = evaluate_loss(
            raw_model, validation_loader, bos, device, cfg.validation_batches
        )
    if distributed:
        value = torch.tensor(
            float("nan") if validation_loss is None else validation_loss,
            dtype=torch.float64,
            device=device,
        )
        dist.broadcast(value, src=0)
        validation_loss = value.item()
    final = output_dir / "foundation.pt"
    if rank == 0:
        atomic_save(
            {
                "model": raw_model.state_dict(),
                "total_steps": completed,
                "target_steps": cfg.steps,
                "config": asdict(cfg),
                "parameters": parameter_count(raw_model),
                "train_loss": losses,
                "validation_loss": validation_loss,
            },
            final,
        )
    distributed_barrier(distributed)
    reason = "target_steps" if completed >= cfg.steps else "time_limit_finalized"
    return final, validation_loss, True, completed, reason


@torch.inference_mode()
def generate(model, tokenizer, records, cfg, device):
    model.eval()
    pad, bos = tokenizer.token_to_id("<pad>"), tokenizer.token_to_id("<bos>")
    items, predictions = list(records.items()), {}
    truncated = 0
    for start in range(0, len(items), cfg.generation_batch_size):
        batch = items[start : start + cfg.generation_batch_size]
        encoded = []
        for _, record in batch:
            ids = tokenizer.encode(record["text"]).ids
            truncated += int(len(ids) > cfg.source_len)
            encoded.append(ids[-cfg.source_len :])
        width = max(map(len, encoded))
        source = torch.full((len(batch), width), pad, dtype=torch.long, device=device)
        source_mask = torch.zeros((len(batch), width), dtype=torch.bool, device=device)
        for row, ids in enumerate(encoded):
            source[row, : len(ids)] = torch.tensor(ids, device=device)
            source_mask[row, : len(ids)] = True
        memory = model.encode(source, source_mask)
        generated = torch.full((len(batch), 1), bos, dtype=torch.long, device=device)
        finished = [None] * len(batch)
        for _ in range(cfg.max_generation_tokens):
            hidden = model.decode(generated, memory, source_mask)
            next_id = model.logits(hidden[:, -1:])[:, -1].argmax(-1, keepdim=True)
            generated = torch.cat((generated, next_id), 1)
            if generated.size(1) % 8 == 0:
                for row in range(len(batch)):
                    words = tokenizer.decode(generated[row, 1:].tolist(), skip_special_tokens=True).split()
                    if len(words) >= cfg.target_words and finished[row] is None:
                        finished[row] = " ".join(words[: cfg.target_words])
                if all(finished):
                    break
        for row, (key, _) in enumerate(batch):
            words = tokenizer.decode(generated[row, 1:].tolist(), skip_special_tokens=True).split()
            if not words:
                words = ["the"]
            if len(words) < cfg.target_words:
                words = (words * math.ceil(cfg.target_words / len(words)))[: cfg.target_words]
            predictions[str(key)] = {"text": finished[row] or " ".join(words[: cfg.target_words])}
        print(f"generated {min(start + len(batch), len(items))}/{len(items)}")
    print(f"prompts truncated to {cfg.source_len} tokens: {truncated}/{len(items)}")
    return predictions


def distributed_generate(model, tokenizer, records, cfg, device, distributed, rank, world_size):
    if not distributed:
        return generate(model, tokenizer, records, cfg, device)
    local_records = dict(list(records.items())[rank::world_size])
    local_predictions = generate(model, tokenizer, local_records, cfg, device)
    gathered = [None] * world_size
    dist.all_gather_object(gathered, local_predictions)
    if rank != 0:
        return None
    predictions = {}
    for shard in gathered:
        predictions.update(shard)
    return predictions


def build_legacy_config(cfg):
    result = legacy.RunConfig(
        steps=cfg.steps,
        train_chars=cfg.train_chars,
        train_tokens=cfg.token_pool,
        seq_len=cfg.source_len,
        batch_size=cfg.batch_size,
        grad_accum=cfg.grad_accum,
    )
    return result


def preparation_signature(cfg):
    return {
        "train_chars": cfg.train_chars,
        "token_pool": cfg.token_pool,
        "tokenizer_chars": cfg.tokenizer_chars,
        "validation_examples": cfg.validation_examples,
        "source_len": cfg.source_len,
        "target_len": cfg.target_len,
        "vocab_size": cfg.vocab_size,
    }


def prepare_data(cfg, output, tokenizer_path, tokenizer_source=None):
    prepared_path = output / "prepared_data.pt"
    expected = preparation_signature(cfg)
    if prepared_path.exists() and tokenizer_path.exists():
        prepared = torch.load(prepared_path, map_location="cpu", weights_only=False)
        if prepared.get("signature") != expected:
            raise ValueError(
                f"{prepared_path} was built with different data settings; use another output "
                "directory or remove that cache"
            )
        return prepared

    texts = legacy.collect_training_text(build_legacy_config(cfg))
    random.Random(SEED).shuffle(texts)
    texts, validation_texts, validation_prompts, validation_truth = make_validation(
        texts, cfg.validation_examples
    )
    if tokenizer_source:
        tokenizer = Tokenizer.from_file(tokenizer_source)
        if Path(tokenizer_source) != tokenizer_path:
            tokenizer.save(str(tokenizer_path))
    else:
        tokenizer = train_tokenizer(texts, tokenizer_path, cfg)
    cfg.vocab_size = tokenizer.get_vocab_size()
    expected["vocab_size"] = cfg.vocab_size
    tokens, spans = tokenize_documents(texts, tokenizer, cfg)
    validation_source_len, validation_target_len = 512, 128
    validation_tokens, validation_spans = tokenize_documents(
        validation_texts, tokenizer, cfg, validation_source_len, validation_target_len
    )
    prepared = {
        "signature": expected,
        "tokens": tokens,
        "spans": spans,
        "validation_tokens": validation_tokens,
        "validation_spans": validation_spans,
        "validation_prompts": validation_prompts,
        "validation_truth": validation_truth,
    }
    atomic_save(prepared, prepared_path)
    (output / "validation_prompts.json").write_text(
        json.dumps(validation_prompts, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    (output / "validation_truth.json").write_text(
        json.dumps(validation_truth, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    return prepared


def main(args):
    session_started = time.perf_counter()
    distributed, rank, world_size, local_rank, device = init_distributed()
    try:
        seed_everything()
        cfg = Config(
            steps=args.steps,
            train_chars=args.train_chars,
            token_pool=args.token_pool,
            validation_examples=args.validation_examples,
            batch_size=args.batch_size,
            grad_accum=args.grad_accum,
            use_global_memory=not args.no_global_memory,
        )
        output = Path(args.output_dir)
        if rank == 0:
            output.mkdir(parents=True, exist_ok=True)
        distributed_barrier(distributed)
        tokenizer_path = output / "legal_bpe_24k.json"

        if args.generate_only:
            if not args.checkpoint or not args.tokenizer:
                raise ValueError("--generate-only requires --checkpoint and --tokenizer")
            tokenizer = Tokenizer.from_file(args.tokenizer)
            saved = torch.load(args.checkpoint, map_location="cpu", weights_only=False)
            cfg = Config(**saved["config"])
            model = LegalEncoderDecoder(cfg).to(device)
            model.load_state_dict(saved["model"])
            prepared_path = output / "prepared_data.pt"
            if args.validation_only and not prepared_path.exists():
                raise ValueError("--validation-only requires prepared_data.pt in --output-dir")
            if prepared_path.exists():
                prepared = torch.load(prepared_path, map_location="cpu", weights_only=False)
                validation_predictions = distributed_generate(
                    model,
                    tokenizer,
                    prepared["validation_prompts"],
                    cfg,
                    device,
                    distributed,
                    rank,
                    world_size,
                )
                if rank == 0:
                    (output / "validation_prediction_foundation.json").write_text(
                        json.dumps(validation_predictions, ensure_ascii=False, indent=2),
                        encoding="utf-8",
                    )
            if args.validation_only:
                if rank == 0:
                    print("created validation_prediction_foundation.json; skipped official test generation")
                return
            test_holder = [legacy.load_test_data() if rank == 0 else None]
            if distributed:
                dist.broadcast_object_list(test_holder, src=0)
            test = test_holder[0]
            predictions = distributed_generate(
                model, tokenizer, test, cfg, device, distributed, rank, world_size
            )
            if rank == 0:
                zip_path = legacy.validate_and_zip(predictions, test, output, "foundation")
                print(f"created {zip_path}")
            return

        if args.checkpoint and not args.tokenizer and not tokenizer_path.exists():
            raise ValueError("An external --checkpoint must be paired with its --tokenizer")
        tokenizer_source = args.tokenizer or (str(tokenizer_path) if tokenizer_path.exists() else None)
        if rank == 0:
            prepare_data(cfg, output, tokenizer_path, tokenizer_source)
        distributed_barrier(distributed)
        prepared = torch.load(
            output / "prepared_data.pt", map_location="cpu", weights_only=False
        )
        tokenizer = Tokenizer.from_file(str(tokenizer_path))
        cfg.vocab_size = tokenizer.get_vocab_size()
        if prepared.get("signature") != preparation_signature(cfg):
            raise ValueError("Prepared-data cache does not match the requested configuration")
        validation_prompts = prepared["validation_prompts"]
        validation_truth = prepared["validation_truth"]

        resume = args.checkpoint
        automatic = output / "foundation_recovery.pt"
        if resume is None and automatic.exists():
            resume = str(automatic)
        completed_steps = 0
        if resume:
            saved = torch.load(resume, map_location="cpu", weights_only=False)
            completed_steps = int(saved.get("completed_steps", saved.get("total_steps", 0)))
            saved_cfg = saved.get("config", {})
            architecture_fields = (
                "vocab_size", "d_model", "heads", "encoder_layers", "decoder_layers",
                "d_ff", "source_len", "target_len", "global_block", "use_global_memory",
            )
            mismatched = [
                name for name in architecture_fields
                if name in saved_cfg and saved_cfg[name] != getattr(cfg, name)
            ]
            if mismatched:
                raise ValueError(f"Checkpoint architecture mismatch: {', '.join(mismatched)}")
            schedule_fields = ("steps", "batch_size", "grad_accum")
            changed_schedule = [
                name for name in schedule_fields
                if name in saved_cfg and saved_cfg[name] != getattr(cfg, name)
            ]
            if changed_schedule:
                raise ValueError(
                    "Checkpoint training schedule mismatch: "
                    f"{', '.join(changed_schedule)}. Resume with the original values."
                )
        if completed_steps >= cfg.steps:
            raise ValueError(
                f"Checkpoint is already at {completed_steps} steps, not below target {cfg.steps}"
            )
        if cfg.grad_accum % world_size:
            raise ValueError("--grad-accum must be divisible by the number of DDP processes")
        local_grad_accum = cfg.grad_accum // world_size
        sample_offset = completed_steps * cfg.batch_size * cfg.grad_accum
        local_samples = (
            (cfg.steps - completed_steps) * cfg.batch_size * local_grad_accum
        )
        samples_per_step = cfg.batch_size * local_grad_accum
        training = ContinuationDataset(
            prepared["tokens"],
            prepared["spans"],
            cfg.source_len,
            cfg.target_len,
            local_samples,
            SEED,
            sample_offset,
            rank,
            world_size,
            samples_per_step,
        )
        validation_source_len, validation_target_len = 512, 128
        validation = ContinuationDataset(
            prepared["validation_tokens"],
            prepared["validation_spans"],
            validation_source_len,
            validation_target_len,
            cfg.validation_batches * cfg.batch_size,
            SEED + 1_000_000,
        )
        loader = torch.utils.data.DataLoader(
            training, batch_size=cfg.batch_size, pin_memory=True
        )
        validation_loader = torch.utils.data.DataLoader(
            validation, batch_size=cfg.batch_size, pin_memory=True
        )
        model = LegalEncoderDecoder(cfg).to(device)
        parameters = parameter_count(model)
        if rank == 0:
            print(
                f"parameters: {parameters:,}; GPUs: {world_size}; "
                f"local accumulation: {local_grad_accum}; global accumulation: {cfg.grad_accum}"
            )
        if not 100_000_000 <= parameters <= 150_000_000:
            raise ValueError("Flagship model must remain inside the recommended 100M-150M range")
        if distributed:
            model = DDP(
                model,
                device_ids=[local_rank],
                output_device=local_rank,
                gradient_as_bucket_view=True,
            )
        checkpoint_path, validation_loss, complete, completed_steps, stop_reason = train(
            model,
            loader,
            validation_loader,
            tokenizer,
            cfg,
            output,
            device,
            resume,
            args.run_steps,
            distributed,
            rank,
            world_size,
            args.max_hours,
            args.generation_reserve_hours,
            session_started,
            args.finalize_on_time_limit,
        )
        model = unwrap_model(model)
        if not complete:
            if rank == 0:
                partial = {
                    "status": "training_incomplete",
                    "stop_reason": stop_reason,
                    "completed_steps": completed_steps,
                    "checkpoint": str(checkpoint_path),
                    "tokenizer": str(tokenizer_path),
                    "prepared_data": str(output / "prepared_data.pt"),
                    "target_steps": cfg.steps,
                    "world_size": world_size,
                }
                (output / "manifest_partial.json").write_text(
                    json.dumps(partial, indent=2), encoding="utf-8"
                )
                print(json.dumps(partial, indent=2))
            return
        validation_predictions = distributed_generate(
            model, tokenizer, validation_prompts, cfg, device, distributed, rank, world_size
        )
        if rank == 0:
            (output / "validation_prediction_foundation.json").write_text(
                json.dumps(validation_predictions, ensure_ascii=False, indent=2),
                encoding="utf-8",
            )
        manifest = {
            "architecture": "12-layer bidirectional encoder + 8-layer causal decoder",
            "original_component": "gated token-level and pooled global cross-attention memory",
            "parameters": parameters,
            "completed_steps": completed_steps,
            "target_steps": cfg.steps,
            "stop_reason": stop_reason,
            "world_size": world_size,
            "training_tokens_processed": completed_steps * cfg.batch_size * cfg.grad_accum * (cfg.source_len + cfg.target_len),
            "validation_loss": validation_loss,
            "training_seconds": time.perf_counter() - session_started,
            "checkpoint": str(checkpoint_path),
            "recovery_checkpoint": str(output / "foundation_recovery.pt"),
            "tokenizer": str(tokenizer_path),
            "submission": None,
            "config": asdict(cfg),
        }
        if args.validation_only:
            if rank == 0:
                (output / "manifest.json").write_text(
                    json.dumps(manifest, indent=2), encoding="utf-8"
                )
                print(json.dumps(manifest, indent=2))
                print("validation-only run complete; skipped official test generation")
            return
        test_holder = [legacy.load_test_data() if rank == 0 else None]
        if distributed:
            dist.broadcast_object_list(test_holder, src=0)
        test = test_holder[0]
        predictions = distributed_generate(
            model, tokenizer, test, cfg, device, distributed, rank, world_size
        )
        if rank != 0:
            return
        zip_path = legacy.validate_and_zip(predictions, test, output, "foundation")
        manifest["submission"] = str(zip_path)
        (output / "manifest.json").write_text(
            json.dumps(manifest, indent=2), encoding="utf-8"
        )
        print(json.dumps(manifest, indent=2))
    finally:
        if distributed and dist.is_initialized():
            dist.destroy_process_group()


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--steps", type=int, default=60_000)
    parser.add_argument("--run-steps", type=int)
    parser.add_argument("--max-hours", type=float)
    parser.add_argument("--generation-reserve-hours", type=float, default=0.0)
    parser.add_argument("--finalize-on-time-limit", action="store_true")
    parser.add_argument("--train-chars", type=int, default=500_000_000)
    parser.add_argument("--token-pool", type=int, default=100_000_000)
    parser.add_argument("--validation-examples", type=int, default=256)
    parser.add_argument("--batch-size", type=int, default=1)
    parser.add_argument("--grad-accum", type=int, default=8)
    parser.add_argument("--output-dir", default="/kaggle/working/phase1_foundation")
    parser.add_argument("--tokenizer")
    parser.add_argument("--checkpoint")
    parser.add_argument("--generate-only", action="store_true")
    parser.add_argument("--validation-only", action="store_true")
    parser.add_argument("--no-global-memory", action="store_true")
    return parser.parse_args()


if __name__ == "__main__":
    main(parse_args())


In [ ]:
%%writefile /kaggle/working/phase2_char_ner.py
"""Exact-character LNER head over the Phase 1 foundation encoder."""

from __future__ import annotations

import argparse
import json
import math
import os
import random
import zipfile
from collections import Counter
from dataclasses import asdict
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from phase1_foundation import Config, LegalEncoderDecoder
from phase1_pipeline import EfficientLM, RunConfig


REPO_ID = "Exploration-Lab/CS-781-Capstone"
FILES = (
    "Phase_2_LNER/Train/fold1-text.json",
    "Phase_2_LNER/Train/fold1-gold.json",
    "Phase_2_LNER/Train/fold2-text.json",
    "Phase_2_LNER/Train/fold2-gold.json",
    "Phase_2_LNER/Train/labels.txt",
    "Phase_2_LNER/Test/fold3-text.json",
)
EXPECTED_LABELS = {
    "APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT",
    "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO",
}


def seed_all(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def find_one(name):
    matches = list(Path("/kaggle/input").rglob(name))
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected exactly one {name}; found {matches}")
    return matches[0]


def safe_load(path):
    return torch.load(path, map_location="cpu", weights_only=True)


def download_data(destination):
    from huggingface_hub import hf_hub_download

    token = os.environ.get("HF_TOKEN")
    if not token:
        raise RuntimeError("Load the Kaggle secret HF into HF_TOKEN first")
    destination.mkdir(parents=True, exist_ok=True)
    result = {}
    for filename in FILES:
        cached = Path(hf_hub_download(
            repo_id=REPO_ID, repo_type="dataset", filename=filename, token=token,
        ))
        target = destination / Path(filename).name
        target.write_bytes(cached.read_bytes())
        result[target.name] = target
    return result


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def texts(raw):
    return {
        str(key): value if isinstance(value, str) else value["text"]
        for key, value in raw.items()
    }


def entity(value):
    if isinstance(value, dict):
        return (
            int(value.get("start", value.get("start_offset"))),
            int(value.get("end", value.get("end_offset"))),
            str(value.get("label", value.get("type"))),
        )
    return int(value[0]), int(value[1]), str(value[2])


def gold(raw):
    result = {}
    for key, value in raw.items():
        if isinstance(value, dict):
            value = value.get("entities", value.get("spans", value.get("annotations")))
        result[str(key)] = [entity(item) for item in value]
    return result


def load_labels(path):
    found = []
    for line in path.read_text(encoding="utf-8").splitlines():
        for chunk in line.split(","):
            candidate = chunk.strip().split()[0] if chunk.strip() else ""
            if candidate in EXPECTED_LABELS:
                found.append(candidate)
    if len(found) != 12 or set(found) != EXPECTED_LABELS:
        raise ValueError(found)
    return found


def tags(labels):
    values = ["O"] + [tag for label in labels for tag in (f"B-{label}", f"I-{label}")]
    return values, {value: index for index, value in enumerate(values)}


def character_labels(text, spans, tag_to_id):
    result = [tag_to_id["O"]] * len(text)
    for start, end, label in sorted(spans):
        if not 0 <= start < end <= len(text):
            raise ValueError((start, end, label, len(text)))
        if any(result[index] != tag_to_id["O"] for index in range(start, end)):
            raise ValueError(f"Overlapping gold span {(start, end, label)}")
        result[start] = tag_to_id[f"B-{label}"]
        for index in range(start + 1, end):
            result[index] = tag_to_id[f"I-{label}"]
    return result


class CharacterWindows(Dataset):
    def __init__(self, documents, tokenizer, max_tokens, stride, annotations=None, tag_to_id=None):
        self.documents = documents
        self.items = []
        step = max_tokens - stride
        for document_id, text in documents.items():
            encoded = tokenizer.encode(text)
            ids = list(encoded.ids)
            offsets = [tuple(map(int, value)) for value in encoded.offsets]
            doc_labels = character_labels(text, annotations[document_id], tag_to_id) if annotations else None
            for token_start in range(0, len(ids), step):
                token_end = min(token_start + max_tokens, len(ids))
                window_offsets = offsets[token_start:token_end]
                if not window_offsets:
                    continue
                char_start = window_offsets[0][0]
                char_end = window_offsets[-1][1]
                if char_end <= char_start:
                    continue
                mapping = [-1] * (char_end - char_start)
                for local_token, (left, right) in enumerate(window_offsets):
                    for position in range(max(left, char_start), min(right, char_end)):
                        mapping[position - char_start] = local_token
                previous = 0
                for index, value in enumerate(mapping):
                    if value >= 0:
                        previous = value
                    else:
                        mapping[index] = previous
                following = len(window_offsets) - 1
                for index in range(len(mapping) - 1, -1, -1):
                    if mapping[index] >= 0:
                        following = mapping[index]
                    else:
                        mapping[index] = following
                self.items.append({
                    "document_id": document_id,
                    "char_start": char_start,
                    "ids": ids[token_start:token_end],
                    "char_ids": [ord(char) % 4096 for char in text[char_start:char_end]],
                    "char_to_token": mapping,
                    "labels": doc_labels[char_start:char_end] if doc_labels is not None else None,
                })
                if token_end == len(ids):
                    break

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        return self.items[index]


def collate(batch):
    if len(batch) != 1:
        raise ValueError("This exact-character recipe requires batch size 1")
    item = batch[0]
    return {
        "document_id": item["document_id"],
        "char_start": item["char_start"],
        "ids": torch.tensor(item["ids"], dtype=torch.long).unsqueeze(0),
        "mask": torch.ones((1, len(item["ids"])), dtype=torch.bool),
        "char_ids": torch.tensor(item["char_ids"], dtype=torch.long).unsqueeze(0),
        "char_to_token": torch.tensor(item["char_to_token"], dtype=torch.long).unsqueeze(0),
        "labels": None if item["labels"] is None else torch.tensor(item["labels"], dtype=torch.long).unsqueeze(0),
    }


class CharacterNER(nn.Module):
    def __init__(self, backbone, tag_count, model_name):
        super().__init__()
        self.backbone = backbone
        self.model_name = model_name
        width = backbone.cfg.d_model
        self.char_embedding = nn.Embedding(4096, 64)
        self.input = nn.Linear(width + 64, 256)
        self.norm1 = nn.LayerNorm(256)
        self.conv1 = nn.Conv1d(256, 256, kernel_size=5, padding=2)
        self.norm2 = nn.LayerNorm(256)
        self.conv2 = nn.Conv1d(256, 256, kernel_size=5, padding=2)
        self.dropout = nn.Dropout(0.15)
        self.classifier = nn.Linear(256, tag_count)

    def head_parameters(self):
        for name, parameter in self.named_parameters():
            if not name.startswith("backbone."):
                yield parameter

    def forward(self, ids, mask, char_ids, char_to_token):
        if self.model_name == "foundation":
            token_hidden = self.backbone.encode(ids, mask)
        else:
            token_hidden = self.backbone.embedding(ids)
            for block in self.backbone.blocks:
                token_hidden = block(token_hidden)
            token_hidden = self.backbone.norm(token_hidden)
        indexes = char_to_token.unsqueeze(-1).expand(-1, -1, token_hidden.size(-1))
        character_hidden = token_hidden.gather(1, indexes)
        x = torch.cat((character_hidden, self.char_embedding(char_ids)), dim=-1)
        x = F.gelu(self.input(x))
        residual = x
        x = self.conv1(self.norm1(x).transpose(1, 2)).transpose(1, 2)
        x = F.gelu(x) + residual
        residual = x
        x = self.conv2(self.norm2(x).transpose(1, 2)).transpose(1, 2)
        return self.classifier(self.dropout(F.gelu(x) + residual))


def decode(tag_ids, tag_names):
    spans = []
    active_label = None
    start = None
    for position, tag_id in enumerate(tag_ids + [0]):
        tag = tag_names[tag_id]
        if tag == "O":
            if active_label is not None:
                spans.append([start, position, active_label])
            active_label, start = None, None
            continue
        prefix, label = tag.split("-", 1)
        if prefix == "B" or label != active_label:
            if active_label is not None:
                spans.append([start, position, active_label])
            active_label, start = label, position
    return spans


def macro_f1(predictions, truth, labels):
    details = {}
    for label in labels:
        predicted = {(doc, s, e) for doc, spans in predictions.items() for s, e, value in spans if value == label}
        actual = {(doc, s, e) for doc, spans in truth.items() for s, e, value in spans if value == label}
        tp = len(predicted & actual)
        precision = tp / len(predicted) if predicted else 0.0
        recall = tp / len(actual) if actual else 0.0
        score = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        details[label] = {"precision": precision, "recall": recall, "f1": score, "support": len(actual)}
    return sum(row["f1"] for row in details.values()) / len(labels), details


@torch.inference_mode()
def score_documents(model, loader, documents, tag_count, device):
    model.eval()
    totals = {key: torch.zeros((len(text), tag_count)) for key, text in documents.items()}
    counts = {key: torch.zeros(len(text)) for key, text in documents.items()}
    for batch in loader:
        ids = batch["ids"].to(device)
        mask = batch["mask"].to(device)
        char_ids = batch["char_ids"].to(device)
        mapping = batch["char_to_token"].to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            logits = model(ids, mask, char_ids, mapping).float().cpu()[0]
        start = batch["char_start"]
        end = start + logits.size(0)
        totals[batch["document_id"]][start:end] += logits
        counts[batch["document_id"]][start:end] += 1
    return totals, counts


def predictions_from_scores(totals, counts, documents, tag_names, o_bias):
    result = {}
    for document_id, text in documents.items():
        scores = totals[document_id] / counts[document_id].clamp_min(1).unsqueeze(1)
        scores[:, 0] -= o_bias
        result[document_id] = decode(scores.argmax(1).tolist(), tag_names)
    return result


def run(args):
    seed_all(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    output = Path(args.output_dir)
    output.mkdir(parents=True, exist_ok=True)
    paths = download_data(output / "data")

    from tokenizers import Tokenizer
    tokenizer_name = "legal_bpe_24k.json" if args.model == "foundation" else "phase1_bpe.json"
    checkpoint_name = "foundation.pt" if args.model == "foundation" else "efficient.pt"
    tokenizer = Tokenizer.from_file(str(find_one(tokenizer_name)))
    checkpoint = safe_load(find_one(checkpoint_name))
    phase1_steps = checkpoint.get("total_steps", checkpoint.get("completed_steps", checkpoint.get("step")))
    if args.model == "foundation":
        cfg = Config(**checkpoint["config"])
        backbone = LegalEncoderDecoder(cfg)
    else:
        cfg = RunConfig(**checkpoint["config"])
        backbone = EfficientLM(cfg)
    backbone.load_state_dict(checkpoint["model"], strict=True)
    if args.model == "foundation":
        for parameter in backbone.decoder.parameters():
            parameter.requires_grad = False
        for parameter in backbone.decoder_norm.parameters():
            parameter.requires_grad = False
    else:
        for block in backbone.blocks:
            block.attention.bidirectional = True

    label_names = load_labels(paths["labels.txt"])
    tag_names, tag_to_id = tags(label_names)
    train_text = texts(read_json(paths["fold1-text.json"]))
    train_gold = gold(read_json(paths["fold1-gold.json"]))
    valid_text = texts(read_json(paths["fold2-text.json"]))
    valid_gold = gold(read_json(paths["fold2-gold.json"]))
    test_text = texts(read_json(paths["fold3-text.json"]))

    train_set = CharacterWindows(train_text, tokenizer, args.max_tokens, args.stride, train_gold, tag_to_id)
    valid_set = CharacterWindows(valid_text, tokenizer, args.max_tokens, args.stride, valid_gold, tag_to_id)
    test_set = CharacterWindows(test_text, tokenizer, args.max_tokens, args.stride)
    train_loader = DataLoader(train_set, batch_size=1, shuffle=True, collate_fn=collate, num_workers=2)
    valid_loader = DataLoader(valid_set, batch_size=1, shuffle=False, collate_fn=collate, num_workers=2)
    test_loader = DataLoader(test_set, batch_size=1, shuffle=False, collate_fn=collate, num_workers=2)

    model = CharacterNER(backbone, len(tag_names), args.model).to(device)
    head = list(model.head_parameters())
    head_ids = {id(parameter) for parameter in head}
    encoder = [parameter for parameter in model.parameters() if parameter.requires_grad and id(parameter) not in head_ids]
    total_parameters = sum(parameter.numel() for parameter in model.parameters())
    trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    print(json.dumps({
        "architecture": f"{args.model}_encoder_plus_exact_character_cnn",
        "phase1_steps": phase1_steps,
        "total_parameters": total_parameters,
        "trainable_parameters": trainable_parameters,
        "train_windows": len(train_set),
        "validation_windows": len(valid_set),
        "test_windows": len(test_set),
        "device": str(device),
    }, indent=2))
    optimizer = torch.optim.AdamW([
        {"params": encoder, "lr": args.backbone_lr},
        {"params": head, "lr": args.head_lr},
    ], weight_decay=0.01)
    scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")

    counts = Counter(tag for item in train_set.items for tag in item["labels"])
    maximum = max(counts.values())
    weights = torch.tensor([
        0.03 if index == 0 else min(10.0, math.sqrt(maximum / max(1, counts[index])))
        for index in range(len(tag_names))
    ], device=device)

    best_score = -1.0
    best_path = output / "best.pt"
    history = []
    for epoch in range(1, args.epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        losses = []
        for step, batch in enumerate(train_loader, 1):
            ids = batch["ids"].to(device)
            mask = batch["mask"].to(device)
            char_ids = batch["char_ids"].to(device)
            mapping = batch["char_to_token"].to(device)
            targets = batch["labels"].to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                logits = model(ids, mask, char_ids, mapping)
                loss = F.cross_entropy(logits.flatten(0, 1), targets.flatten(), weight=weights)
                loss = loss / args.grad_accum
            scaler.scale(loss).backward()
            if step % args.grad_accum == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(encoder + head, 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
            losses.append(float(loss.detach()) * args.grad_accum)

        totals, window_counts = score_documents(model, valid_loader, valid_text, len(tag_names), device)
        score, details, selected_bias = -1.0, None, 0.0
        for bias in (0.0, 0.5, 1.0, 1.5, 2.0, 3.0):
            predicted = predictions_from_scores(totals, window_counts, valid_text, tag_names, bias)
            candidate, candidate_details = macro_f1(predicted, valid_gold, label_names)
            if candidate > score:
                score, details, selected_bias = candidate, candidate_details, bias
        row = {"epoch": epoch, "loss": sum(losses) / len(losses), "strict_macro_f1": score, "o_bias": selected_bias}
        history.append(row)
        print(json.dumps(row), flush=True)
        if score > best_score:
            best_score = score
            torch.save({"model": model.state_dict(), "score": score, "o_bias": selected_bias, "per_label": details}, best_path)

    saved = safe_load(best_path)
    model.load_state_dict(saved["model"], strict=True)
    totals, window_counts = score_documents(model, test_loader, test_text, len(tag_names), device)
    predicted = predictions_from_scores(totals, window_counts, test_text, tag_names, saved["o_bias"])
    prediction_path = output / "prediction.json"
    prediction_path.write_text(json.dumps(predicted, ensure_ascii=False), encoding="utf-8")
    archive_path = output / f"submission_phase2_{args.model}_character.zip"
    with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
        archive.write(prediction_path, arcname="prediction.json")
    with zipfile.ZipFile(archive_path) as archive:
        assert archive.namelist() == ["prediction.json"]
    (output / "training_log.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
    (output / "validation_metrics.json").write_text(json.dumps({
        "strict_macro_f1": saved["score"], "o_bias": saved["o_bias"], "per_label": saved["per_label"],
    }, indent=2), encoding="utf-8")
    (output / "manifest.json").write_text(json.dumps({
        "architecture": f"{args.model}_encoder_plus_exact_character_cnn",
        "phase1_config": asdict(cfg),
        "phase1_steps": phase1_steps,
        "total_parameters": total_parameters,
        "trainable_parameters": trainable_parameters,
        "backbone_learning_rate": args.backbone_lr,
        "head_learning_rate": args.head_lr,
        "epochs": args.epochs,
        "max_tokens": args.max_tokens,
        "stride": args.stride,
        "seed": args.seed,
        "train_windows": len(train_set),
        "validation_windows": len(valid_set),
        "test_windows": len(test_set),
        "best_validation_f1": saved["score"],
        "submission": str(archive_path),
    }, indent=2), encoding="utf-8")
    print("UPLOAD:", archive_path)


def self_test():
    names, mapping = tags(["COURT"])
    labels = character_labels("xHigh Court!", [(1, 11, "COURT")], mapping)
    assert decode(labels, names) == [[1, 11, "COURT"]]
    print("Exact-character self-check passed")


def arguments():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", choices=("foundation", "efficient"), default="foundation")
    parser.add_argument("--output-dir", default="/kaggle/working/phase2_foundation_character")
    parser.add_argument("--epochs", type=int, default=10)
    parser.add_argument("--max-tokens", type=int, default=384)
    parser.add_argument("--stride", type=int, default=96)
    parser.add_argument("--grad-accum", type=int, default=2)
    parser.add_argument("--backbone-lr", type=float, default=2e-5)
    parser.add_argument("--head-lr", type=float, default=1e-3)
    parser.add_argument("--seed", type=int, default=230338)
    parser.add_argument("--self-test", action="store_true")
    return parser.parse_args()


if __name__ == "__main__":
    args = arguments()
    if args.self_test:
        self_test()
    else:
        run(args)


In [ ]:
!python -m py_compile /kaggle/working/phase1_pipeline.py /kaggle/working/phase1_foundation.py /kaggle/working/phase2_char_ner.py
!python /kaggle/working/phase2_char_ner.py --self-test


In [ ]:
!python /kaggle/working/phase2_char_ner.py --model efficient --epochs 10 --max-tokens 384 --stride 96 --grad-accum 2 --backbone-lr 0.00003 --head-lr 0.001 --output-dir /kaggle/working/phase2_efficient_character


In [ ]:
from pathlib import Path
import json
root = Path('/kaggle/working/phase2_efficient_character')
print(json.dumps(json.loads((root / 'validation_metrics.json').read_text()), indent=2))
print('UPLOAD:', root / 'submission_phase2_efficient_character.zip')
